In [ ]:
# first try 

In [ ]:
# !pip install ultralytics easyocr opencv-python-headless

In [ ]:
# from roboflow import Roboflow
# from ultralytics import YOLO

# # 1. Download Dataset from Roboflow
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("ROBOFLOW_KEY")

# rf = Roboflow(api_key=secret_value_0)
# project = rf.workspace("aryans-workspace-qdqdp").project("scoreboard_dataset_final")
# dataset = project.version(1).download("yolo26")

# # 2. Train YOLOv26 Model
# model = YOLO('yolov26n.pt') 

# # Train for a quick experiment (adjust epochs for better convergence)
# results = model.train(
#     data=f"{dataset.location}/data.yaml",
#     epochs=20,
#     imgsz=640,
#     batch=8,
#     device='0,1' # Uses Kaggle GPU
# )

# # 3. Load the trained weights for inference
# trained_model = YOLO('/kaggle/working/runs/detect/train/weights/best.pt')
# print("Model training complete and weights loaded.")

In [ ]:
# import cv2
# import easyocr
# import re
# import torch
# from datetime import timedelta

# class CricketHighlightPipeline:
#     def __init__(self, yolo_model, video_path, max_duration_mins=30, sample_fps=1):
#         self.yolo_model = yolo_model
#         self.video_path = video_path
#         self.max_duration_sec = max_duration_mins * 60
#         self.sample_fps = sample_fps
#         # self.reader = easyocr.Reader(['en'], gpu=True)
#         self.reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
        
#         # Tracking state
#         self.last_valid_score = {"runs": None, "wickets": None}
#         self.highlights = []

#     def preprocess_ocr_text(self, text):
#         """Hardcode common OCR errors"""
#         replacements = {
#             'O': '0', 'o': '0', 'l': '1', 'I': '1', 'i': '1', 
#             'S': '5', 's': '5', 'B': '8', 'Z': '2', 'z': '2'
#         }
#         for k, v in replacements.items():
#             text = text.replace(k, v)
#         # Remove anything that isn't a digit, slash, or dash
#         text = re.sub(r'[^\d/-]', '', text)
#         return text

#     def extract_score(self, text):
#         """Extract runs and wickets using regex"""
#         cleaned_text = self.preprocess_ocr_text(text)
#         # Matches formats like 145/3 or 145-3
#         match = re.search(r'(\d{1,3})\s*[/|-]\s*(\d{1,2})', cleaned_text)
        
#         if match:
#             runs = int(match.group(1))
#             wickets = int(match.group(2))
#             return runs, wickets
#         return None, None

#     def validate_temporal_consistency(self, current_runs, current_wickets):
#         """Ensure the score changes are logically possible in cricket"""
#         if self.last_valid_score["runs"] is None:
#             self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
#             return False, None

#         prev_runs = self.last_valid_score["runs"]
#         prev_wickets = self.last_valid_score["wickets"]

#         run_diff = current_runs - prev_runs
#         wicket_diff = current_wickets - prev_wickets

#         event_type = None

#         # Logical Constraints: 
#         # Runs shouldn't jump by more than 7 per ball. Wickets shouldn't jump by more than 1.
#         # Scores shouldn't go backwards (ignoring penalties/reviews for this basic pipeline).
#         if 0 <= run_diff <= 7 and 0 <= wicket_diff <= 1:
#             if run_diff == 4:
#                 event_type = "4 Runs (Boundary)"
#             elif run_diff == 6:
#                 event_type = "6 Runs (Six)"
#             elif wicket_diff == 1:
#                 event_type = "Wicket"
            
#             # Update state
#             self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
        
#         return (event_type is not None), event_type

#     def process_video(self):
#         cap = cv2.VideoCapture(self.video_path)
#         cap = cv2.VideoCapture(self.video_path)

#         if not cap.isOpened():
#             raise ValueError(f"Error opening video: {self.video_path}")
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         if fps <= 0:
#             raise ValueError("Invalid FPS from video")
        
#         frame_interval = max(1, int(fps / self.sample_fps))
        
#         frame_count = 0
        
#         while cap.isOpened():
#             ret, frame = cap.read()
#             if not ret:
#                 break
                
#             current_time_sec = frame_count / fps
            
#             # Stop if we reach the max duration (e.g., 30 mins)
#             if current_time_sec > self.max_duration_sec:
#                 break

#             # Process 1 frame per second (based on sample_fps)
#             if frame_count % frame_interval == 0:
#                 # 1. Detect Lower Third
#                 results = self.yolo_model.predict(frame, conf=0.5, verbose=False)
#                 if not results or results[0].boxes is None:
#                     continue
                
#                 for box in results[0].boxes:
#                     x1, y1, x2, y2 = map(int, box.xyxy[0])
#                     h, w = frame.shape[:2]
#                     x1, y1 = max(0, x1), max(0, y1)
#                     x2, y2 = min(w, x2), min(h, y2)
                    
#                     # 2. Crop detected region
#                     roi = frame[y1:y2, x1:x2]
                    
#                     if roi.size > 0:
#                         # Convert to grayscale for better OCR
#                         gray_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
                        
#                         # 3. Apply OCR
#                         ocr_results = self.reader.readtext(gray_roi, detail=0)
#                         combined_text = " ".join(ocr_results)
                        
#                         # 4. Extract and Validate
#                         runs, wickets = self.extract_score(combined_text)
                        
#                         if runs is not None and wickets is not None:
#                             is_highlight, event = self.validate_temporal_consistency(runs, wickets)
                            
#                             # 5. Record Highlight
#                             if is_highlight:
#                                 start_t = max(0, int(current_time_sec) - 10)
#                                 end_t = int(current_time_sec) + 5
                                
#                                 self.highlights.append({
#                                     "event": event,
#                                     "timestamp_sec": int(current_time_sec),
#                                     "interval": f"{timedelta(seconds=start_t)} to {timedelta(seconds=end_t)}",
#                                     "score": f"{runs}/{wickets}"
#                                 })
            
#             frame_count += 1

#         cap.release()
#         return self.highlights

In [ ]:
# # Path to the Kaggle dataset video
# # Update this path based on where the other user's dataset is mounted in your Kaggle environment
# VIDEO_PATH = "/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4" 

# print("Starting video processing pipeline...")
# pipeline = CricketHighlightPipeline(
#     yolo_model=trained_model,
#     video_path=VIDEO_PATH,
#     max_duration_mins=30,  # Only process first 30 mins
#     sample_fps=1           # 1 Frame per second for optimal performance vs accuracy
# )

# detected_highlights = pipeline.process_video()

# if not detected_highlights:
#     print("No highlights detected.")
# else:
#     for hl in detected_highlights:
#         print(f"[{hl['interval']}] Event: {hl['event']} | Score Updated to: {hl['score']}")

In [ ]:
# second try 

In [ ]:
# !pip install -q ultralytics easyocr opencv-python-headless tqdm

# import os
# import cv2
# import re
# import torch
# import numpy as np
# from datetime import timedelta
# from ultralytics import YOLO
# import easyocr
# from tqdm.notebook import tqdm

# # Define paths
# WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
# VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

# # GPU Selection: Utilizing both T4s
# device_yolo = "cuda:0" if torch.cuda.device_count() > 0 else "cpu"
# device_ocr = "cuda:1" if torch.cuda.device_count() > 1 else device_yolo

# print(f"YOLO running on: {device_yolo}")
# print(f"EasyOCR running on: {device_ocr}")

In [ ]:
# # Initialize YOLOv8
# trained_model = YOLO(WEIGHTS_PATH)
# trained_model.to(device_yolo)

# # Initialize EasyOCR with the second GPU
# # We use the specific device index to ensure parallel utilization
# reader = easyocr.Reader(['en'], gpu=True) 
# # Note: EasyOCR handles internal torch device management, 
# # but if errors occur on T4, we fallback to default GPU logic.

# print("Models loaded and ready for inference.")

In [ ]:
# class CricketHighlightPipeline:
#     def __init__(self, yolo_model, ocr_reader, video_path, max_duration_mins=None, sample_fps=1):
#         self.yolo_model = yolo_model
#         self.reader = ocr_reader
#         self.video_path = video_path
#         self.sample_fps = sample_fps
        
#         # --- FIX: Handle None for Full Match ---
#         if max_duration_mins is not None:
#             self.max_duration_sec = max_duration_mins * 60
#         else:
#             self.max_duration_sec = float('inf') # Infinity means no time limit
            
#         self.last_valid_score = {"runs": None, "wickets": None}
#         self.highlights = []

#     def preprocess_ocr_text(self, text):
#         replacements = {'O': '0', 'o': '0', 'l': '1', 'I': '1', 'i': '1', 'S': '5', 's': '5', 'B': '8', 'Z': '2', 'z': '2'}
#         for k, v in replacements.items():
#             text = text.replace(k, v)
#         return re.sub(r'[^\d/-]', '', text)

#     def extract_score(self, text):
#         cleaned_text = self.preprocess_ocr_text(text)
#         match = re.search(r'(\d{1,3})\s*[/|-]\s*(\d{1,2})', cleaned_text)
#         if match:
#             return int(match.group(1)), int(match.group(2))
#         return None, None

#     def validate_and_update(self, current_runs, current_wickets):
#         if self.last_valid_score["runs"] is None:
#             self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
#             return False, None

#         prev_runs = self.last_valid_score["runs"]
#         prev_wickets = self.last_valid_score["wickets"]
        
#         run_diff = current_runs - prev_runs
#         wicket_diff = current_wickets - prev_wickets

#         # --- NEW: INNINGS RESET LOGIC ---
#         # If the current score is very low (e.g. < 5) and the previous score was high (> 50),
#         # treat it as a new innings and reset the baseline.
#         if current_runs < 5 and prev_runs > 50:
#             print(f"🔄 Innings break detected! Resetting baseline to {current_runs}/{current_wickets}")
#             self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
#             return False, None

#         # Standard Cricket Logic
#         if 0 <= run_diff <= 7 and 0 <= wicket_diff <= 1:
#             event_type = None
#             if run_diff == 4: event_type = "4 Runs (Boundary)"
#             elif run_diff == 6: event_type = "6 Runs (Six)"
#             elif wicket_diff == 1: event_type = "Wicket"
            
#             if run_diff > 0 or wicket_diff > 0:
#                 self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            
#             return (event_type is not None), event_type
        
#         return False, None

#     def process_video(self):
#         cap = cv2.VideoCapture(self.video_path)
#         fps = cap.get(cv2.CAP_PROP_FPS)
#         total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
#         limit_frames = total_frames if self.max_duration_sec == float('inf') else int(self.max_duration_sec * fps)
#         total_to_process = min(total_frames, limit_frames)
        
#         frame_interval = max(1, int(fps / self.sample_fps))
        
#         # --- FIX: tqdm.notebook with refresh control ---
#         pbar = tqdm(total=total_to_process, desc="Analyzing Match", leave=True, mininterval=1.0)
        
#         frame_count = 0
#         while cap.isOpened():
#             # Use 'grab' for skipping to save CPU, 'read' only for the frames we need
#             if frame_count % frame_interval == 0:
#                 ret, frame = cap.read()
#             else:
#                 ret = cap.grab()
#                 frame_count += 1
#                 if frame_count % 10 == 0: # Update bar every 10 frames to keep it smooth
#                     pbar.update(10)
#                 continue

#             current_time_sec = frame_count / fps
#             if not ret or current_time_sec > self.max_duration_sec:
#                 break
                
#             # --- Inference Logic ---
#             h, w = frame.shape[:2]
#             lower_third = frame[int(h * 0.75):h, 0:w]
#             results = self.yolo_model.predict(lower_third, conf=0.6, verbose=False, device=0)
            
#             if results and len(results[0].boxes) > 0:
#                 x1, y1, x2, y2 = map(int, results[0].boxes[0].xyxy[0])
#                 roi = lower_third[y1:y2, x1:x2]
#                 if roi.size > 0:
#                     ocr_res = self.reader.readtext(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), detail=0)
#                     runs, wickets = self.extract_score(" ".join(ocr_res))
#                     if runs is not None:
#                         is_hl, event = self.validate_and_update(runs, wickets)
#                         if is_hl:
#                             ts = int(current_time_sec)
#                             self.highlights.append({
#                                 "event": event, "ts": ts, "score": f"{runs}/{wickets}",
#                                 "interval": f"{timedelta(seconds=max(0, ts-10))} - {timedelta(seconds=ts+5)}"
#                             })
            
#             frame_count += 1
#             pbar.update(1)

#         cap.release()
#         pbar.close() # Cleanly close the bar
#         return self.highlights

In [ ]:
# # Paths (Verify these in your Kaggle sidebar)
# WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
# VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

# print("🚀 Initializing Full Match Pipeline...")

# pipeline = CricketHighlightPipeline(
#     yolo_model=trained_model,
#     ocr_reader=reader,
#     video_path=VIDEO_PATH,
#     max_duration_mins=None,  # Set to None for the whole match
#     sample_fps=1
# )

# detected_highlights = pipeline.process_video()

# print("\n" + "="*40)
# print("🏏 FULL MATCH HIGHLIGHTS SUMMARY")
# print("="*40)

# if not detected_highlights:
#     print("No highlights detected. Double-check your scoreboard detection confidence.")
# else:
#     for hl in detected_highlights:
#         print(f"[{hl['interval']}] {hl['event']} | Score: {hl['score']}")

In [ ]:
# third try 

In [ ]:
!pip install -q ultralytics easyocr opencv-python-headless tqdm

import os
import cv2
import re
import torch
import numpy as np
from datetime import timedelta
from ultralytics import YOLO
import easyocr
from tqdm.notebook import tqdm

WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

device_yolo = "cuda:0" if torch.cuda.device_count() > 0 else "cpu"
device_ocr = "cuda:1" if torch.cuda.device_count() > 1 else device_yolo

print(f"YOLO running on: {device_yolo}")
print(f"EasyOCR running on: {device_ocr}")

In [ ]:
trained_model = YOLO(WEIGHTS_PATH)
trained_model.to(device_yolo)

reader = easyocr.Reader(['en'], gpu=True)

print("Models loaded and ready for inference.")

In [ ]:
class CricketHighlightPipeline:
    def __init__(self, yolo_model, ocr_reader, video_path, max_duration_mins=None, sample_fps=1, persistence=3):
        self.yolo_model = yolo_model
        self.reader = ocr_reader
        self.video_path = video_path
        self.sample_fps = sample_fps
        self.persistence_limit = persistence
        
        if max_duration_mins is not None:
            self.max_duration_sec = max_duration_mins * 60
        else:
            self.max_duration_sec = float('inf')
            
        self.last_valid_score = {"runs": None, "wickets": None}
        self.highlights = []
        self.score_buffer = []
        self.innings_active = False

    def preprocess_ocr_text(self, text):
        replacements = {'O': '0', 'o': '0', 'l': '1', 'I': '1', 'i': '1', 'S': '5', 's': '5', 'B': '8', 'Z': '2', 'z': '2'}
        for k, v in replacements.items():
            text = text.replace(k, v)
        return re.sub(r'[^\d/-]', '', text)

    def extract_score(self, text):
        cleaned_text = self.preprocess_ocr_text(text)
        match = re.search(r'(\d{1,3})\s*[/|-]\s*(\d{1,2})', cleaned_text)
        if match:
            return int(match.group(1)), int(match.group(2))
        return None, None

    def get_stable_score(self, runs, wickets):
        self.score_buffer.append((runs, wickets))
        if len(self.score_buffer) > self.persistence_limit:
            self.score_buffer.pop(0)
        
        if len(self.score_buffer) == self.persistence_limit and len(set(self.score_buffer)) == 1:
            return self.score_buffer[0]
        return None, None

    def validate_and_update(self, current_runs, current_wickets):
        if current_runs == 0 and current_wickets == 0:
            is_prev_valid = self.last_valid_score["runs"] is not None and self.last_valid_score["runs"] > 0
            is_initial = self.last_valid_score["runs"] is None
            
            if (is_initial or is_prev_valid) and not self.innings_active:
                self.last_valid_score = {"runs": 0, "wickets": 0}
                self.innings_active = True
                return True, "New Innings Detected"
            elif self.innings_active:
                return False, None

        if current_runs > 0 or current_wickets > 0:
            self.innings_active = False

        if self.last_valid_score["runs"] is None:
            self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            return False, None

        prev_runs = self.last_valid_score["runs"]
        prev_wickets = self.last_valid_score["wickets"]
        
        run_diff = current_runs - prev_runs
        wicket_diff = current_wickets - prev_wickets

        if 0 <= run_diff <= 7 and 0 <= wicket_diff <= 1:
            event_type = None
            if run_diff == 4: event_type = "Four"
            elif run_diff == 6: event_type = "Six"
            elif wicket_diff == 1: event_type = "Wicket"
            
            if run_diff > 0 or wicket_diff > 0:
                self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            
            return (event_type is not None), event_type
        
        return False, None

    def process_video(self):
        cap = cv2.VideoCapture(self.video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        limit_frames = total_frames if self.max_duration_sec == float('inf') else int(self.max_duration_sec * fps)
        total_to_process = min(total_frames, limit_frames)
        
        frame_interval = max(1, int(fps / self.sample_fps))
        
        pbar = tqdm(total=total_to_process, desc="Analyzing Match", leave=True, mininterval=1.0)
        
        frame_count = 0
        while cap.isOpened():
            if frame_count % frame_interval == 0:
                ret, frame = cap.read()
            else:
                ret = cap.grab()
                frame_count += 1
                if frame_count % 10 == 0:
                    pbar.update(10)
                continue

            current_time_sec = frame_count / fps
            if not ret or current_time_sec > self.max_duration_sec:
                break
                
            h, w = frame.shape[:2]
            lower_third = frame[int(h * 0.75):h, 0:w]
            results = self.yolo_model.predict(lower_third, conf=0.45, verbose=False, device=0)
            
            if results and len(results[0].boxes) > 0:
                boxes = sorted(results[0].boxes, key=lambda x: x.conf[0].item(), reverse=True)
                x1, y1, x2, y2 = map(int, boxes[0].xyxy[0])
                roi = lower_third[y1:y2, x1:x2]
                
                if roi.size > 0:
                    ocr_res = self.reader.readtext(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), detail=0)
                    raw_runs, raw_wickets = self.extract_score(" ".join(ocr_res))
                    
                    if raw_runs is not None:
                        stable_runs, stable_wickets = self.get_stable_score(raw_runs, raw_wickets)
                        
                        if stable_runs is not None:
                            is_hl, event = self.validate_and_update(stable_runs, stable_wickets)
                            if is_hl:
                                ts = int(current_time_sec)
                                start_time = max(0, ts - 15)
                                end_time = ts + 10
                                duration = end_time - start_time
                                
                                hl_data = {
                                    "event": event,
                                    "ts": ts,
                                    "score": f"{stable_runs}/{stable_wickets}",
                                    "duration": duration,
                                    "timestamps": f"[{timedelta(seconds=start_time)}] - [{timedelta(seconds=end_time)}]"
                                }
                                
                                if event == "New Innings Detected":
                                    print(f"\n{event} | Timestamps: {hl_data['timestamps']} | Score: {hl_data['score']}")
                                self.highlights.append(hl_data)
            
            frame_count += 1
            pbar.update(1)

        cap.release()
        pbar.close()
        return self.highlights

In [ ]:
WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

print("Initializing Full Match Pipeline...")

pipeline = CricketHighlightPipeline(
    yolo_model=trained_model,
    ocr_reader=reader,
    video_path=VIDEO_PATH,
    max_duration_mins=None,
    sample_fps=1,
    persistence=3 
)

detected_highlights = pipeline.process_video()

print("\n" + "="*40)
print("🏏 FULL MATCH HIGHLIGHTS SUMMARY")
print("="*40)

if not detected_highlights:
    print("No highlights detected. Double-check your scoreboard detection confidence.")
else:
    for hl in detected_highlights:
        if hl['event'] != "New Innings Detected":
            print(f"Highlight: {hl['event']} | Duration: {hl['duration']}s | Timestamps: {hl['timestamps']} | Score: {hl['score']}")

In [ ]:
import os
import subprocess

# Define directories
output_dir = '/kaggle/working/results'
clips_dir = os.path.join(output_dir, 'temp_clips')
os.makedirs(clips_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'output.mp4')
concat_list_path = os.path.join(output_dir, 'concat_list.txt')

if not detected_highlights:
    print("No highlights detected. Skipping video generation.")
else:
    print(f"Extracting {len(detected_highlights)} clips using FFmpeg (Stream Copy)...")
    valid_clips = []
    
    for i, hl in enumerate(detected_highlights):
        # We recalculate the exact seconds from the timestamp (ts)
        ts = hl['ts']
        start_time = max(0, ts - 15)
        duration = 25 # (ts + 10) - (ts - 15)
        
        clip_path = os.path.join(clips_dir, f"clip_{i:03d}.mp4")
        
        # FFmpeg command: -ss before -i enables highly optimized fast-seeking
        cmd = [
            'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
            '-ss', str(start_time), 
            '-i', VIDEO_PATH, 
            '-t', str(duration), 
            '-c', 'copy', 
            clip_path
        ]
        
        # Execute the extraction
        subprocess.run(cmd)
        
        if os.path.exists(clip_path):
            valid_clips.append(clip_path)
            print(f"✅ Extracted: {hl['event']} -> {clip_path}")

    # Create the text file required for FFmpeg's concat demuxer
    print("\nPreparing to merge clips...")
    with open(concat_list_path, 'w') as f:
        for clip in valid_clips:
            # FFmpeg requires absolute paths inside the text file
            f.write(f"file '{os.path.abspath(clip)}'\n")

    # FFmpeg command: Concatenate all clips in the list together instantly
    concat_cmd = [
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
        '-f', 'concat', 
        '-safe', '0', 
        '-i', concat_list_path, 
        '-c', 'copy', 
        output_path
    ]
    
    subprocess.run(concat_cmd)
    
    if os.path.exists(output_path):
        print(f"\n🎉 SUCCESS! Final highlight video saved to: {output_path}")
        print("Note: You can download this file from the Kaggle 'Output' sidebar under /working/results/")
    else:
        print("\n❌ Error: Failed to merge videos. Check FFmpeg logs.")

In [ ]:
from IPython.display import FileLink

# Generate a direct download link for the final video
print("✅ Video ready! Click the link below to download:")
display(FileLink('results/output.mp4'))

In [ ]:
!pip install -q ultralytics easyocr opencv-python-headless tqdm

import os
import cv2
import re
import torch
import numpy as np
import subprocess
from datetime import timedelta
from ultralytics import YOLO
import easyocr
from tqdm.notebook import tqdm
from IPython.display import FileLink, display


# -------------------------------
# PATHS
# -------------------------------
WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'


# -------------------------------
# DEVICE SETUP
# -------------------------------
device_yolo = "cuda:0" if torch.cuda.device_count() > 0 else "cpu"
device_ocr = "cuda:1" if torch.cuda.device_count() > 1 else device_yolo

print(f"YOLO running on: {device_yolo}")
print(f"EasyOCR running on: {device_ocr}")

trained_model = YOLO(WEIGHTS_PATH)
trained_model.to(device_yolo)

reader = easyocr.Reader(['en'], gpu=True)

print("Models loaded and ready for inference.")


# -------------------------------
# PIPELINE CLASS
# -------------------------------
class CricketHighlightPipeline:
    def __init__(self, yolo_model, ocr_reader, video_path,
                 max_duration_mins=None, sample_fps=1, persistence=3):

        self.yolo_model = yolo_model
        self.reader = ocr_reader
        self.video_path = video_path
        self.sample_fps = sample_fps
        self.persistence_limit = persistence

        self.max_duration_sec = max_duration_mins * 60 if max_duration_mins else float('inf')

        self.last_valid_score = {"runs": None, "wickets": None}
        self.highlights = []
        self.score_buffer = []
        self.innings_active = False

        # 🔥 Motion-related variables
        self.prev_frame = None
        self.motion_buffer = []
        self.motion_threshold = 2.0
        self.motion_persistence = 3


    # -------------------------------
    # OCR PROCESSING
    # -------------------------------
    def preprocess_ocr_text(self, text):
        replacements = {
            'O': '0','o': '0','l': '1','I': '1','i': '1',
            'S': '5','s': '5','B': '8','Z': '2','z': '2'
        }
        for k, v in replacements.items():
            text = text.replace(k, v)
        return re.sub(r'[^\d/-]', '', text)

    def extract_score(self, text):
        cleaned_text = self.preprocess_ocr_text(text)
        match = re.search(r'(\d{1,3})\s*[/|-]\s*(\d{1,2})', cleaned_text)
        if match:
            return int(match.group(1)), int(match.group(2))
        return None, None

    def get_stable_score(self, runs, wickets):
        self.score_buffer.append((runs, wickets))

        if len(self.score_buffer) > self.persistence_limit:
            self.score_buffer.pop(0)

        if len(self.score_buffer) == self.persistence_limit and len(set(self.score_buffer)) == 1:
            return self.score_buffer[0]

        return None, None


    # -------------------------------
    # MOTION (OPTICAL FLOW)
    # -------------------------------
    def compute_optical_flow_intensity(self, prev_gray, curr_gray):
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, curr_gray,
            None,
            0.5, 3, 15, 3, 5, 1.2, 0
        )

        magnitude, _ = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        return np.mean(magnitude)

    def get_motion_intensity(self, frame, prev_frame):
        if prev_frame is None:
            return 0.0

        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        curr_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        return self.compute_optical_flow_intensity(prev_gray, curr_gray)

    def get_stable_motion(self, motion_val):
        self.motion_buffer.append(motion_val)

        if len(self.motion_buffer) > self.motion_persistence:
            self.motion_buffer.pop(0)

        if len(self.motion_buffer) == self.motion_persistence:
            return np.mean(self.motion_buffer)

        return None


    # -------------------------------
    # EVENT VALIDATION
    # -------------------------------
    def validate_and_update(self, current_runs, current_wickets):

        if current_runs == 0 and current_wickets == 0:
            if not self.innings_active:
                self.last_valid_score = {"runs": 0, "wickets": 0}
                self.innings_active = True
                return True, "New Innings Detected"
            return False, None

        if current_runs > 0 or current_wickets > 0:
            self.innings_active = False

        if self.last_valid_score["runs"] is None:
            self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            return False, None

        prev_runs = self.last_valid_score["runs"]
        prev_wickets = self.last_valid_score["wickets"]

        run_diff = current_runs - prev_runs
        wicket_diff = current_wickets - prev_wickets

        if 0 <= run_diff <= 7 and 0 <= wicket_diff <= 1:

            event_type = None
            if run_diff == 4: event_type = "Four"
            elif run_diff == 6: event_type = "Six"
            elif wicket_diff == 1: event_type = "Wicket"

            if run_diff > 0 or wicket_diff > 0:
                self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}

            return (event_type is not None), event_type

        return False, None


    # -------------------------------
    # MAIN PROCESSING
    # -------------------------------
    def process_video(self):

        cap = cv2.VideoCapture(self.video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        frame_interval = max(1, int(fps / self.sample_fps))
        pbar = tqdm(total=total_frames, desc="Analyzing Match")

        frame_count = 0

        while cap.isOpened():

            if frame_count % frame_interval == 0:
                ret, frame = cap.read()
            else:
                ret = cap.grab()
                frame_count += 1
                continue

            if not ret:
                break

            # ---------------- MOTION ----------------
            motion_val = self.get_motion_intensity(frame, self.prev_frame)
            stable_motion = self.get_stable_motion(motion_val)
            self.prev_frame = frame.copy()

            current_time_sec = frame_count / fps

            h, w = frame.shape[:2]
            lower_third = frame[int(h * 0.75):h, :]

            results = self.yolo_model.predict(lower_third, conf=0.45, verbose=False)

            if results and len(results[0].boxes) > 0:

                box = results[0].boxes[0]
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                roi = lower_third[y1:y2, x1:x2]

                if roi.size > 0:
                    ocr_res = self.reader.readtext(cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY), detail=0)
                    runs, wickets = self.extract_score(" ".join(ocr_res))

                    if runs is not None:
                        stable_runs, stable_wickets = self.get_stable_score(runs, wickets)

                        if stable_runs is not None:
                            is_hl, event = self.validate_and_update(stable_runs, stable_wickets)

                            # 🔥 Motion fallback
                            if not is_hl and stable_motion is not None:
                                if self.motion_threshold < stable_motion < 15:
                                    is_hl = True
                                    event = "High Motion Event"

                            if is_hl:
                                ts = int(current_time_sec)
                                start = max(0, ts - 15)
                                end = ts + 10

                                self.highlights.append({
                                    "event": event,
                                    "ts": ts,
                                    "score": f"{stable_runs}/{stable_wickets}",
                                    "timestamps": f"[{timedelta(seconds=start)} - {timedelta(seconds=end)}]"
                                })

            frame_count += 1
            pbar.update(1)

        cap.release()
        pbar.close()

        return self.highlights


# -------------------------------
# RUN PIPELINE
# -------------------------------
pipeline = CricketHighlightPipeline(trained_model, reader, VIDEO_PATH)

highlights = pipeline.process_video()

print("\n🏏 HIGHLIGHTS")
for h in highlights:
    print(h)


# -------------------------------
# VIDEO GENERATION (FFmpeg)
# -------------------------------
output_dir = '/kaggle/working/results'
clips_dir = os.path.join(output_dir, 'temp_clips')
os.makedirs(clips_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'output.mp4')
concat_file = os.path.join(output_dir, 'concat.txt')

valid_clips = []

for i, hl in enumerate(highlights):
    ts = hl['ts']
    start = max(0, ts - 15)

    clip_path = os.path.join(clips_dir, f"clip_{i}.mp4")

    cmd = [
        'ffmpeg','-y','-loglevel','error',
        '-ss', str(start),
        '-i', VIDEO_PATH,
        '-t','25',
        '-c','copy',
        clip_path
    ]
    subprocess.run(cmd)

    if os.path.exists(clip_path):
        valid_clips.append(clip_path)

with open(concat_file, 'w') as f:
    for clip in valid_clips:
        f.write(f"file '{clip}'\n")

subprocess.run([
    'ffmpeg','-y','-loglevel','error',
    '-f','concat','-safe','0',
    '-i', concat_file,
    '-c','copy',
    output_path
])

print("\n✅ Final video ready:")
display(FileLink(output_path))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.7 MB/s eta 0:00:0000:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO running on: cuda:0
EasyOCR running on: cuda:1
Models loaded and ready for inference.


Analyzing Match:   0%|          | 0/370914 [00:00<?, ?it/s]